In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Load anomaly and driver data
anomaly = pd.read_csv("...", parse_dates=["date"])
driver = pd.read_csv("...", parse_dates=["date"])


In [ ]:
anomaly["region_id"].unique()


In [ ]:
f = anomaly.copy()
dr = driver.copy()

latest_date = f["date"].max()
latest = f[f["date"] == latest_date]


In [ ]:
# Trend variance in each region
trend_region = f.pivot_table(index="date", columns="region_id", values="variance", aggfunc="sum")
trend_region.plot(marker="o", figsize=(10, 4), title="Variance trend by region")
plt.axhline(0, color="black", linewidth=0.8)
plt.show()


In [ ]:
# Trend Actual vs Budget
trend = latest.groupby("region_id")[["value_actual", "value_budget"]].sum()
trend.plot(kind="bar", figsize=(8, 4), title=f"Actual vs Budget ({latest_date:%Y-%m})")
plt.xticks(rotation=0)
plt.xlabel("Region")
plt.show()


In [ ]:
# Identify top 3 contributors
top = (latest.groupby("account_id", as_index=False)["variance"].sum()
       .assign(abs_var=lambda x: x["variance"].abs())
       .nlargest(3, "abs_var"))
colors = ["green" if v >= 0 else "red" for v in top["variance"]]
top.plot(kind="bar", x="account_id", y="variance", color=colors, legend=False, figsize=(8, 4), title=f"Top 3 variance contributors ({latest_date:%Y-%m})")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=0)
plt.xlabel("Account")
plt.show()


In [ ]:
# Revenue driver factors
dd = dr[dr["date"] == latest_date].groupby("driver", as_index=False)["impact"].sum()
colors = ["green" if v >= 0 else "red" for v in dd["impact"]]
dd.plot(kind="bar", x="driver", y="impact", color=colors, legend=False, title="Revenue drivers")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=0)
plt.xlabel("Driver")
plt.show()


In [ ]:
# Anomaly details
latest[latest["anomaly"].astype(bool)].sort_values("variance")


In [ ]:
f.groupby("date")["anomaly"].sum()
